# 08. Fixing people counted twice by the tiles

Notebook 07 added tiled detection for larger groups. A real photo exposed a bug: 11 cricketers standing in a line were counted as **18**. They were too tall to fit in one tile, so many were found as a top half and a bottom half, and each half was counted.

**Fix:** when two boxes are both cut by a tile edge and line up across it, join them into one person. The alignment limit (0.5) was chosen on tuning photos only (26 Open Images, 75 COCO). This notebook re-runs the app's real `count_auto` on all test sets to check that nothing got worse.

In [1]:
import os, sys, time
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "app")
import numpy as np
import pandas as pd
import scipy.io as sio
from PIL import Image
import crowd_model as cm

RESULTS = os.path.join("data", "cache", "join_fix_predictions.csv")
LOG = os.path.join("data", "cache", "join_fix_progress.log")
print("alignment limit:", cm.JOIN_ALIGN)

alignment limit: 0.5


In [2]:
def shanghai(part):
    root = os.path.join("data", "shanghaitech", "ShanghaiTech", f"part_{part}", "test_data")
    rows = []
    for name in sorted(os.listdir(os.path.join(root, "images"))):
        gt = sio.loadmat(os.path.join(root, "ground-truth", "GT_" + name.replace(".jpg", ".mat")))
        rows.append((f"ShanghaiTech {part}", os.path.join(root, "images", name), len(gt["image_info"][0, 0][0, 0][0])))
    return rows

oi = pd.read_csv("data/openimages/large_groups.csv").query("split == 'test'")
coco = pd.read_csv("data/coco/small_groups.csv").query("split == 'test'")
photos = [("Open Images 14-38", "data/openimages/images/" + f, int(c)) for f, c in zip(oi.file_name, oi.true_count)]
photos += [("COCO 1-13", "data/coco/images/" + f, int(c)) for f, c in zip(coco.file_name, coco.true_count)]
photos += shanghai("B") + shanghai("A")
print(pd.Series([p[0] for p in photos]).value_counts().to_string())

ShanghaiTech B       316
ShanghaiTech A       182
COCO 1-13            150
Open Images 14-38     26


In [3]:
done = pd.read_csv(RESULTS) if os.path.exists(RESULTS) else pd.DataFrame(columns=["set", "path", "true", "pred", "method"])
seen = set(done.path)
models = {k: cm.load_model(k) for k in cm.MODELS}
detector = cm.load_detector()
start = time.time()
new = 0
for s, path, true in photos:
    if path in seen:
        continue
    r = cm.count_auto(cm.prepare_image(Image.open(path))[0], models, detector)
    row = pd.DataFrame([{"set": s, "path": path, "true": true, "pred": r.count, "method": r.method}])
    row.to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    new += 1
    if new % 20 == 0:
        with open(LOG, "a") as f:
            f.write(f"{len(seen) + new}/{len(photos)} photos, {time.time() - start:.0f}s\n")
results = pd.read_csv(RESULTS)
print(len(results), "photos counted")

674 photos counted


In [4]:
before = {"Open Images 14-38": 18.2, "COCO 1-13": 3.43, "ShanghaiTech B": 16.3, "ShanghaiTech A": 86.9}
results["error"] = (results.pred - results.true).abs()
table = results.groupby("set").agg(photos=("error", "size"), mae=("error", "mean"), median=("error", "median"),
                                   within_2=("error", lambda e: (e <= 2).mean() * 100))
table.insert(1, "mae_before", pd.Series(before))
table.loc[list(before)].round(2)

,photos,mae_before,mae,median,within_2
set,,,,,
Open Images 14-38,26,18.20,18.53,4.50,19.23
COCO 1-13,150,3.43,3.45,1.00,71.33
ShanghaiTech B,316,16.30,16.24,8.61,15.51
ShanghaiTech A,182,86.90,86.94,58.08,1.65


## What this shows

- **The bug is reduced, not gone:** the photo of 11 players went from 18 to 9.
- **There is a small cost:** on the 26 larger-group test photos the average error went from 18.2 to 18.53, the median from 3.5 to 4.5, counts within 2 people from 27% to 19%, and photos off by 30 or more from 3 to 4.
- **Other sets barely moved:** COCO 3.43 to 3.45 (within 2 people: 72% to 71%), ShanghaiTech B 16.3 to 16.24, ShanghaiTech A stayed at 86.9.
- **Why it shipped anyway:** cutting people in half is a visible error on ordinary photos, and the measured cost is small. This was a judgement call, because the fix did not meet the rule of leaving larger groups no worse.
- **These test photos have now been seen several times**, so these are not clean held-out scores.